# PALSYN XLSX → CSV 변환 + Duration Outlier 제거

이 노트북은 PALSYN 생성 결과(`*.xlsx`)를 평가용 CSV로 변환하면서, 동일 case 안에서 발생한 비정상적인 timestamp jump 때문에 case duration이 수십 년으로 튀는 케이스를 제거합니다.

기본 설정은 **case duration 기준 상위 1% 제거 + 7일 초과 case 제거 중 더 엄격한 기준**입니다. 원본 synthetic 파일은 보존하고, cleaned CSV와 diagnostics만 새로 저장합니다.

In [ ]:
# ============================================================
# 0. Imports
# ============================================================
from pathlib import Path
import re
import zipfile
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

In [ ]:
# ============================================================
# 1. Config
# ============================================================
# PALSYN xlsx files가 있는 폴더
INPUT_DIR = Path("./")

# 출력 폴더
OUT_DIR = Path("./palsyn_cleaned_csv")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 입력 파일 패턴
# 예: 41mimicel(1).xlsx, 42mimicel(1).xlsx ...
INPUT_PATTERN = "*mimicel*.xlsx"

# 표준 컬럼명
CASE_COL = "stay_id"
ACT_COL = "activity"
TIME_COL = "timestamps"

# PALSYN 원본 컬럼명 후보
CASE_CANDIDATES = ["case:concept:name", "case_id", "case", "stay_id", "Case ID", "caseid"]
ACT_CANDIDATES = ["concept:name", "activity", "Activity", "event", "event_name"]
TIME_CANDIDATES = ["time:timestamp", "timestamp", "timestamps", "time", "Timestamp"]

# Outlier 제거 방식
# - quantile: 각 파일별 case duration의 상위 OUTLIER_Q 초과 case 제거
# - absolute: MAX_DURATION_HOURS 초과 case 제거
# - both: quantile과 absolute 중 더 엄격한 기준 적용
FILTER_MODE = "quantile"
OUTLIER_Q = 0.99
MAX_DURATION_HOURS = 168  # absolute/both에서 사용. 168h = 7일

# duration이 음수/NaN인 case 제거 여부
DROP_INVALID_DURATION = True

# single-event case(duration=0)는 유지할지 여부
KEEP_SINGLE_EVENT_CASES = True

# 결과 zip 생성
MAKE_ZIP = True

In [ ]:
# ============================================================
# 2. Helper functions
# ============================================================
def find_col(df, candidates, kind):
    for c in candidates:
        if c in df.columns:
            return c
    lowered = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lowered:
            return lowered[c.lower()]
    raise ValueError(f"{kind} column not found. Available columns: {list(df.columns)}")


def extract_seed(path: Path):
    m = re.search(r"(\d+)", path.stem)
    return int(m.group(1)) if m else None


def normalize_palsyn_columns(df: pd.DataFrame) -> pd.DataFrame:
    case_src = find_col(df, CASE_CANDIDATES, "case")
    act_src = find_col(df, ACT_CANDIDATES, "activity")
    time_src = find_col(df, TIME_CANDIDATES, "timestamp")

    out = df[[case_src, act_src, time_src]].copy()
    out = out.rename(columns={case_src: CASE_COL, act_src: ACT_COL, time_src: TIME_COL})

    out[CASE_COL] = out[CASE_COL].astype(str)
    out[ACT_COL] = out[ACT_COL].astype(str)
    out[TIME_COL] = pd.to_datetime(out[TIME_COL], utc=True, errors="coerce")

    out = out.dropna(subset=[CASE_COL, ACT_COL, TIME_COL])
    out = out.sort_values([CASE_COL, TIME_COL, ACT_COL]).reset_index(drop=True)
    return out


def compute_case_duration(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(CASE_COL)[TIME_COL].agg(["min", "max", "count"]).reset_index()
    g["duration_hours"] = (g["max"] - g["min"]).dt.total_seconds() / 3600.0
    return g


def choose_threshold(duration_df: pd.DataFrame) -> float:
    valid = duration_df["duration_hours"].replace([np.inf, -np.inf], np.nan).dropna()
    if len(valid) == 0:
        return np.nan

    q_thr = float(valid.quantile(OUTLIER_Q))

    if FILTER_MODE == "quantile":
        return q_thr
    elif FILTER_MODE == "absolute":
        return float(MAX_DURATION_HOURS)
    elif FILTER_MODE == "both":
        return min(q_thr, float(MAX_DURATION_HOURS))
    else:
        raise ValueError("FILTER_MODE must be one of: quantile, absolute, both")


def clean_duration_outliers(df: pd.DataFrame):
    dur = compute_case_duration(df)
    threshold = choose_threshold(dur)

    valid_mask = pd.Series(True, index=dur.index)

    if DROP_INVALID_DURATION:
        valid_mask &= dur["duration_hours"].notna()
        valid_mask &= np.isfinite(dur["duration_hours"])
        valid_mask &= dur["duration_hours"] >= 0

    if not KEEP_SINGLE_EVENT_CASES:
        valid_mask &= dur["count"] > 1

    if not np.isnan(threshold):
        valid_mask &= dur["duration_hours"] <= threshold

    valid_cases = set(dur.loc[valid_mask, CASE_COL])
    removed_cases = set(dur[CASE_COL]) - valid_cases

    clean_df = df[df[CASE_COL].isin(valid_cases)].copy()
    removed_df = dur[dur[CASE_COL].isin(removed_cases)].copy().sort_values("duration_hours", ascending=False)

    diagnostics = {
        "n_events_before": len(df),
        "n_cases_before": df[CASE_COL].nunique(),
        "n_events_after": len(clean_df),
        "n_cases_after": clean_df[CASE_COL].nunique(),
        "n_cases_removed": len(removed_cases),
        "removed_case_ratio": len(removed_cases) / max(1, df[CASE_COL].nunique()),
        "duration_threshold_hours": threshold,
        "duration_min_before": dur["duration_hours"].min(),
        "duration_median_before": dur["duration_hours"].median(),
        "duration_mean_before": dur["duration_hours"].mean(),
        "duration_q95_before": dur["duration_hours"].quantile(0.95),
        "duration_q99_before": dur["duration_hours"].quantile(0.99),
        "duration_max_before": dur["duration_hours"].max(),
    }

    if len(clean_df) > 0:
        dur_after = compute_case_duration(clean_df)
        diagnostics.update({
            "duration_min_after": dur_after["duration_hours"].min(),
            "duration_median_after": dur_after["duration_hours"].median(),
            "duration_mean_after": dur_after["duration_hours"].mean(),
            "duration_q95_after": dur_after["duration_hours"].quantile(0.95),
            "duration_q99_after": dur_after["duration_hours"].quantile(0.99),
            "duration_max_after": dur_after["duration_hours"].max(),
        })
    else:
        diagnostics.update({
            "duration_min_after": np.nan,
            "duration_median_after": np.nan,
            "duration_mean_after": np.nan,
            "duration_q95_after": np.nan,
            "duration_q99_after": np.nan,
            "duration_max_after": np.nan,
        })

    return clean_df, removed_df, diagnostics

In [ ]:
# ============================================================
# 3. Convert all PALSYN XLSX files and remove outlier cases
# ============================================================
files = sorted(INPUT_DIR.glob(INPUT_PATTERN))
files = [f for f in files if not f.name.startswith("~$")]

print(f"Found {len(files)} files")
for f in files:
    print(" -", f)

all_diag = []

for path in files:
    seed = extract_seed(path)
    print("\n" + "="*80)
    print(f"Processing: {path.name} | seed={seed}")

    raw = pd.read_excel(path)
    df = normalize_palsyn_columns(raw)
    clean_df, removed_df, diag = clean_duration_outliers(df)

    diag["file"] = path.name
    diag["seed"] = seed
    diag["filter_mode"] = FILTER_MODE
    diag["outlier_q"] = OUTLIER_Q
    diag["max_duration_hours_config"] = MAX_DURATION_HOURS
    all_diag.append(diag)

    seed_str = str(seed) if seed is not None else path.stem

    # 평가 코드 호환용 표준 CSV
    clean_csv = OUT_DIR / f"palsyn_synthetic_data_seed{seed_str}.csv"
    clean_df.to_csv(clean_csv, index=False, encoding="utf-8-sig")

    # XES 스타일 컬럼명 유지 버전도 저장
    xes_style = clean_df.rename(columns={
        CASE_COL: "case:concept:name",
        ACT_COL: "concept:name",
        TIME_COL: "time:timestamp",
    })
    xes_csv = OUT_DIR / f"{seed_str}mimicel_cleaned_xes_style.csv"
    xes_style.to_csv(xes_csv, index=False, encoding="utf-8-sig")

    # 제거된 case 목록 저장
    removed_csv = OUT_DIR / f"removed_cases_seed{seed_str}.csv"
    removed_df.to_csv(removed_csv, index=False, encoding="utf-8-sig")

    print(f"Saved cleaned: {clean_csv.name}")
    print(f"Removed cases: {diag['n_cases_removed']} / {diag['n_cases_before']} ({diag['removed_case_ratio']:.2%})")
    print(f"Duration before: median={diag['duration_median_before']:.2f}h, mean={diag['duration_mean_before']:.2f}h, max={diag['duration_max_before']:.2f}h")
    print(f"Duration after : median={diag['duration_median_after']:.2f}h, mean={diag['duration_mean_after']:.2f}h, max={diag['duration_max_after']:.2f}h")

diag_df = pd.DataFrame(all_diag)
diag_path = OUT_DIR / "palsyn_cleaning_diagnostics.csv"
diag_df.to_csv(diag_path, index=False, encoding="utf-8-sig")

print("\nDiagnostics saved:", diag_path)
display(diag_df)

In [ ]:
# ============================================================
# 4. Quick check: before/after summary table
# ============================================================
summary_cols = [
    "seed", "n_cases_before", "n_cases_after", "n_cases_removed", "removed_case_ratio",
    "duration_median_before", "duration_mean_before", "duration_max_before",
    "duration_median_after", "duration_mean_after", "duration_max_after",
    "duration_threshold_hours"
]

summary = diag_df[summary_cols].copy()
summary = summary.sort_values("seed")
display(summary)

In [ ]:
# ============================================================
# 5. Zip outputs
# ============================================================
if MAKE_ZIP:
    zip_path = OUT_DIR.parent / "palsyn_cleaned_csv_outputs.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(OUT_DIR.glob("*")):
            zf.write(p, arcname=p.name)
    print("Created:", zip_path)

## 출력 파일

- `palsyn_synthetic_data_seedXX.csv`: 평가 노트북에 바로 넣는 표준 컬럼명 버전
  - `stay_id`, `activity`, `timestamps`
- `XXmimicel_cleaned_xes_style.csv`: XES 스타일 컬럼명 유지 버전
  - `case:concept:name`, `concept:name`, `time:timestamp`
- `removed_cases_seedXX.csv`: 제거된 이상 case 목록
- `palsyn_cleaning_diagnostics.csv`: seed별 제거 전/후 duration 요약
- `palsyn_cleaned_csv_outputs.zip`: 전체 결과 압축 파일

평가 노트북에서는 `palsyn_synthetic_data_seed41.csv` ~ `palsyn_synthetic_data_seed45.csv`를 PALSYN synthetic input으로 사용하면 됩니다.